# 12 — Cost / performance analysis across TPU versions

**Learning goal.** Pick the right TPU for the job. We sweep model size and batch size across (v4, v5e, v5p, v6e) and look at cost per sample (and intuitively, cost per token).

**What you'll observe.**
- A cost-per-sample plot vs model size for each TPU version.
- A cost-per-sample plot vs batch size for each TPU version.
- The **utilisation-adjusted** cost — what you'd be paying if you only achieved X% MXU util.

**Knobs to tune.**
- `hourly_usd_per_chip` — set per-version from cloud.google.com/tpu/pricing (these are placeholders).
- The simulated `step_time_s` model — here we approximate it from `total_flops / peak_bf16_tflops / mxu_util`.
- `batch_sizes`, `hidden_sizes`.

**Failure modes to watch.**
- Pricing changes — never trust hardcoded defaults. Always check the public page before you cite numbers.
- Cost-per-sample alone is misleading: a "cheap" chip you can't fit your model on is infinitely expensive.
- A high cost-per-sample at low MXU util is the diagnostic for "this chip can do more — fix your pipeline before fixing the bill".

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Helper: estimate step time from total FLOPs / chip peak

In [ ]:
from cloud_tpu_lab.src.common.cost import CostInputs, estimate_cost
from cloud_tpu_lab.src.tpu_versions.cloud_tpu_catalog import get_spec, list_versions
from cloud_tpu_lab.src.model_examples.tiny_mlp_jax import build_tiny_mlp_graph
from cloud_tpu_lab.src.xla_sim.lowering import lower_to_hlo

# Placeholder hourly rates — update from cloud.google.com/tpu/pricing.
HOURLY_USD = {
    "v4":  3.22,
    "v5e": 1.20,
    "v5p": 4.00,
    "v6e": 2.70,
}

def step_time_s(total_flops: int, peak_bf16_tflops: float, mxu_util: float) -> float:
    peak_flops = peak_bf16_tflops * 1e12 * max(mxu_util, 1e-3)
    return total_flops / peak_flops

def cost_per_sample(version: str, batch: int, hidden: int, num_layers: int,
                    mxu_util: float, n_steps: int = 1000):
    spec = get_spec(version)
    graph = build_tiny_mlp_graph(batch_size=batch, hidden_size=hidden,
                                 num_layers=num_layers)
    module = lower_to_hlo(graph)
    t = step_time_s(module.total_flops(), spec.peak_bf16_tflops, mxu_util)
    cost = estimate_cost(CostInputs(
        chip_count=1, n_steps=n_steps,
        step_time_s=t,
        hourly_usd_per_chip=HOURLY_USD[version],
        samples_per_step=batch,
        utilization=mxu_util,
    ))
    return cost.cost_per_sample_usd, cost

# Quick sanity check
cps, _ = cost_per_sample("v5e", batch=32, hidden=512, num_layers=4, mxu_util=0.4)
print("v5e example cost_per_sample_usd =", cps)

## Sweep cost-per-sample vs model size

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hidden_sizes = [128, 256, 512, 1024, 2048]
mxu_util = 0.45
batch = 32

fig = plt.figure(figsize=(8, 5))
for v in list_versions():
    ys = []
    for h in hidden_sizes:
        cps, _ = cost_per_sample(v, batch=batch, hidden=h, num_layers=4,
                                 mxu_util=mxu_util)
        ys.append(cps)
    plt.plot(hidden_sizes, ys, marker="o", label=v)

plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("hidden size (model dim)")
plt.ylabel("cost per sample (USD, log)")
plt.title(f"Cost per sample vs model size  (batch={batch}, mxu_util={mxu_util})")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb12_cost_vs_model_size.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Sweep cost-per-sample vs batch size

In [ ]:
batches = [1, 2, 4, 8, 16, 32, 64, 128, 256]
hidden = 1024

fig = plt.figure(figsize=(8, 5))
for v in list_versions():
    ys = []
    for b in batches:
        cps, _ = cost_per_sample(v, batch=b, hidden=hidden, num_layers=4,
                                 mxu_util=mxu_util)
        ys.append(cps)
    plt.plot(batches, ys, marker="o", label=v)

plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("batch size")
plt.ylabel("cost per sample (USD, log)")
plt.title(f"Cost per sample vs batch  (hidden={hidden}, mxu_util={mxu_util})")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb12_cost_vs_batch.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Utilisation-adjusted cost

`CostReport.utilization_adjusted_usd` answers: "If I'm only achieving X% MXU utilisation, what would the same wall-clock cost have been at 100%?" It's the gap you're leaving on the table.

In [ ]:
hidden = 1024
batch = 32

fig = plt.figure(figsize=(8, 5))
for v in list_versions():
    utils = []
    paid_usd = []
    adj_usd = []
    for u in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        cps, cost = cost_per_sample(v, batch=batch, hidden=hidden,
                                    num_layers=4, mxu_util=u)
        utils.append(u)
        paid_usd.append(cost.total_run_usd)
        adj_usd.append(cost.utilization_adjusted_usd)
    plt.plot(utils, adj_usd, marker="o", label=f"{v} (utilization-adjusted)")
    plt.plot(utils, paid_usd, marker="x", linestyle="--", alpha=0.5,
             label=f"{v} (paid)")

plt.xlabel("MXU utilisation")
plt.ylabel("total run USD (1000 steps)")
plt.title("Paid vs utilisation-adjusted cost — gap = money left on the table")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb12_util_adjusted_cost.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## How to read these plots

- **Cost per sample vs model size**: the cheapest chip per sample is rarely the same across model sizes. v5e wins for small models; v5p / v6e tend to win for large ones, where their bigger HBM + higher FLOPS turn into shorter step times.
- **Cost per sample vs batch**: tiny batches usually look expensive because per-sample fixed overhead dominates. The curve flattens once you saturate the chip.
- **Utilisation-adjusted**: a flat dashed line vs a rising solid line shows you the absolute payment is the same (wall clock × hourly rate), but the **adjusted** number rises as utilisation drops — that's the cost-of-being-bottlenecked.

## Always sanity-check pricing

The `HOURLY_USD` dict at the top of this notebook is a **placeholder**. Real prices vary by:

- Region and zone.
- Commitment (on-demand vs 1-yr / 3-yr).
- Spot / preemptible vs on-demand.

Look up the current rate at https://cloud.google.com/tpu/pricing before quoting numbers in a doc or a billing review.

## Exercises

1. Update `HOURLY_USD` with the *current* rates from cloud.google.com/tpu/pricing. Replot. Does the ranking change?
2. Add a model size for `hidden_size=4096`. Does v5e still appear? If not, why might it OOM (cross-reference notebook 08)?
3. Add a `cost_per_token` plot by setting `tokens_per_step = batch * seq_len` for a transformer-ish workload.
4. Plot cost per sample at fixed `mxu_util` but varying `chip_count` from 1..32, assuming strong scaling. At what chip count does v6e become cheapest?
5. Imagine you have a $1000 budget and 24 wall-clock hours. Which TPU/chip-count combination trains the *largest* model? Define "largest" via HBM headroom and use notebook 08 to check feasibility.